In [0]:
from pyspark.sql.functions import (col, avg, sum, count, when, round,
    lit, coalesce, stddev, lag)
from pyspark.sql.window import Window

# ── Step 1: Read from Gold fact table ─────────────────────────────────
fact = spark.table("mini_project_team_a3t7.gold.daily_inventory_fact")

# ── Step 2: Window spec for rolling calculations ───────────────────────
w30  = (Window.partitionBy("store_id", "product_id")
              .orderBy("inventory_date")
              .rowsBetween(-30, 0))    # last 30 days
w90  = (Window.partitionBy("store_id", "product_id")
              .orderBy("inventory_date")
              .rowsBetween(-90, 0))    # last 90 days

# ── Step 3: Inventory KPIs ─────────────────────────────────────────────
inv_kpis = (fact
    # KPI 1: Inventory Turnover Ratio = COGS sold / avg inventory value
    .withColumn("cogs_30d",
        sum(col("units_sold") * col("cost_price")).over(w30))
    .withColumn("avg_stock_value_30d",
        avg("stock_value").over(w30))
    .withColumn("inventory_turnover_ratio",
        round(when(col("avg_stock_value_30d") > 0,
                   col("cogs_30d") / col("avg_stock_value_30d"))
              .otherwise(lit(0)), 3))

    # KPI 2: Overstock Risk Index = current_stock / (30-day avg daily sales × 30)
    .withColumn("avg_daily_sales_30d",
        avg("units_sold").over(w30))
    .withColumn("overstock_risk_index",
        round(when(col("avg_daily_sales_30d") > 0,
                   col("current_stock") /
                   (col("avg_daily_sales_30d") * 30))
              .otherwise(lit(999)), 2))   # 999 = no sales data

    # KPI 3: Dead Stock Probability
    # Units sold in last 90 days = 0 but current_stock > 0
    .withColumn("units_sold_90d",
        sum("units_sold").over(w90))
    .withColumn("is_dead_stock",
        (col("units_sold_90d") == 0) & (col("current_stock") > 0))

    # KPI 4: Days of Inventory on Hand (already in fact table — use rolling version)
    .withColumn("avg_days_on_hand_30d",
        round(avg(when(col("days_on_hand") < 999,
                       col("days_on_hand"))).over(w30), 1))

    # Risk classification
    .withColumn("inventory_health",
        when(col("is_stockout"),                             "Critical — Stockout")
       .when(col("is_dead_stock"),                           "Critical — Dead Stock")
       .when(col("overstock_risk_index") > 1.5,             "Warning — Overstock")
       .when(col("is_low_stock"),                            "Warning — Low Stock")
       .otherwise("Healthy"))

    .select(
        "store_id", "product_id", "inventory_date", "category",
        "inventory_turnover_ratio", "overstock_risk_index",
        "is_dead_stock", "units_sold_90d",
        "avg_days_on_hand_30d", "avg_daily_sales_30d",
        "current_stock", "available_stock", "stock_value",
        "is_stockout", "is_low_stock", "inventory_health"
    )
)

# ── Step 4: Write to Gold ──────────────────────────────────────────────
(inv_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("inventory_date")
    .saveAsTable("mini_project_team_a3t7.gold.inventory_kpis")
)
# print("Inventory health distribution:")
# inv_kpis.groupBy("inventory_health").count().show()

In [0]:
inv_kpis.display()

In [0]:
"""
build_inventory_kpis.py
=======================
Builds gold.inventory_kpis — rolling inventory health KPIs.

Source : mini_project_team_a3t7.gold.fact_inventory_full
Grain  : store_id + product_id + snapshot_date

Columns changed from original:
  daily_inventory_fact  →  fact_inventory_full     (table name)
  inventory_date        →  snapshot_date           (date column)
  current_stock         →  current_stock_qty       (stock column)
  available_stock       →  available_stock_qty     (available stock column)
  stock_value           →  derived: current_stock_qty * cost_price  (not stored, computed)
  days_on_hand          →  stock_cover_days        (pre-computed in fact table)
  is_stockout           →  stockout_flag == 1      (flag column)
  is_low_stock          →  reorder_flag == 1       (flag column)
"""

from pyspark.sql.functions import (
    col, avg, sum, count, when, round,
    lit, coalesce, stddev, lag
)
from pyspark.sql.window import Window

# ── Step 1: Read from Gold fact table ─────────────────────────────────
fact = spark.table("mini_project_team_a3t7.gold.fact_inventory_full")

# ── Step 2: Derive stock_value upfront ────────────────────────────────
# fact_inventory_full does not store stock_value as a column
fact = fact.withColumn(
    "stock_value",
    col("current_stock_qty") * col("cost_price")
)

# ── Step 3: Window specs for rolling calculations ──────────────────────
w30 = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date")           # was inventory_date
             .rowsBetween(-30, 0))

w90 = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date")           # was inventory_date
             .rowsBetween(-90, 0))

# ── Step 4: Inventory KPIs ─────────────────────────────────────────────
inv_kpis = (fact

    # ── KPI 1: Inventory Turnover Ratio ───────────────────────────────
    # = COGS sold over 30 days / avg inventory value over 30 days
    # Higher = stock is moving fast (good)
    # Lower  = stock is sitting idle (capital locked up)
    .withColumn("cogs_30d",
        sum(col("units_sold") * col("cost_price")).over(w30))
    .withColumn("avg_stock_value_30d",
        avg("stock_value").over(w30))
    .withColumn("inventory_turnover_ratio",
        round(
            when(col("avg_stock_value_30d") > 0,
                 col("cogs_30d") / col("avg_stock_value_30d"))
            .otherwise(lit(0)),
            3
        )
    )

    # ── KPI 2: Overstock Risk Index ───────────────────────────────────
    # = current_stock_qty / (30-day avg daily sales × 30)
    # > 1.5 → holding 1.5x more stock than needed for 30 days (overstock)
    # < 0.5 → holding less than 15 days of stock (risk of stockout)
    # 999   → no sales in last 30 days (dead stock candidate)
    .withColumn("avg_daily_sales_30d",
        avg("units_sold").over(w30))
    .withColumn("overstock_risk_index",
        round(
            when(col("avg_daily_sales_30d") > 0,
                 col("current_stock_qty") /           # was current_stock
                 (col("avg_daily_sales_30d") * 30))
            .otherwise(lit(999)),
            2
        )
    )

    # ── KPI 3: Dead Stock Detection ───────────────────────────────────
    # Dead stock = zero units sold in last 90 days but stock still sitting
    # These products are costing storage space with zero revenue return
    .withColumn("units_sold_90d",
        sum("units_sold").over(w90))
    .withColumn("is_dead_stock",
        (col("units_sold_90d") == 0) & (col("current_stock_qty") > 0))  # was current_stock

    # ── KPI 4: Avg Days on Hand (30-day rolling) ──────────────────────
    # stock_cover_days is already computed in fact_inventory_full
    # (was days_on_hand in original — same concept, different name)
    # Filter out nulls (rows where units_sold = 0, cover = null)
    .withColumn("avg_days_on_hand_30d",
        round(
            avg(
                when(col("stock_cover_days").isNotNull(),
                     col("stock_cover_days"))          # was days_on_hand < 999
            ).over(w30),
            1
        )
    )

    # ── KPI 5: Stock Value at Risk ────────────────────────────────────
    # New KPI — value of inventory sitting in dead/overstock products
    # Helps quantify the financial impact of poor inventory management
    .withColumn("stock_value_at_risk",
        round(
            when(
                col("is_dead_stock") |
                (col("overstock_risk_index") > 1.5),
                col("stock_value")
            ).otherwise(lit(0)),
            2
        )
    )

    # ── Risk classification ───────────────────────────────────────────
    # Priority order: stockout > dead stock > overstock > low stock > healthy
    # stockout_flag == 1  replaces is_stockout boolean
    # reorder_flag == 1   replaces is_low_stock boolean
    .withColumn("inventory_health",
        when(col("stockout_flag") == 1,              "Critical — Stockout")   
       .when(col("is_dead_stock"),                   "Critical — Dead Stock")
       .when(col("overstock_risk_index") > 1.5,      "Warning — Overstock")
       .when(col("reorder_flag") == 1,               "Warning — Low Stock")   
       .otherwise("Healthy")
    )

    .select(
        # Keys
        "store_id",
        "product_id",
        "snapshot_date",               
        "category",
        "subcategory",
        "brand",

        # Rolling KPIs
        "inventory_turnover_ratio",
        "overstock_risk_index",
        "avg_daily_sales_30d",
        "avg_days_on_hand_30d",         
        "units_sold_90d",
        "cogs_30d",
        "avg_stock_value_30d",

        # Dead stock
        "is_dead_stock",

        # Stock columns (correct names from fact_inventory_full)
        "current_stock_qty",            
        "available_stock_qty",          
        "reserved_stock_qty",
        "stock_value",                  
        "stock_value_at_risk",          

        # Flags (integer 0/1 in fact_inventory_full, not booleans)
        "stockout_flag",               
        "reorder_flag",                 
        "overstock_flag",

        # Health label
        "inventory_health",
    )
)

# ── Step 5: Write to Gold ──────────────────────────────────────────────
(
    inv_kpis.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")           
    .saveAsTable("mini_project_team_a3t7.gold.inventory_kpis1")
)
